# Multi-Agent Workflow — Tool Setup

This notebook sets up a custom **Tavily web search tool** for use with the `openai-agents-sdk`.

No agents are created in this step — this is purely tool setup. Once agents are built (in a later step), they will use the `gpt-5-mini` model.

In [1]:
import os
from typing import Annotated

from dotenv import load_dotenv
from tavily import TavilyClient
from agents import function_tool

load_dotenv()

TAVILY_API_KEY = os.environ["TAVILY_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # picked up automatically by the Agents SDK

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# Sanity check that keys loaded, without ever printing the actual values
print("TAVILY_API_KEY loaded:", bool(TAVILY_API_KEY))
print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))

TAVILY_API_KEY loaded: True
OPENAI_API_KEY loaded: True


## Custom Tavily web search tool

Wrapped with `@function_tool` so it can be handed to an `Agent`'s `tools=[...]` list later.

In [2]:
@function_tool
def web_search(
    query: Annotated[str, "The search query to look up on the web"],
    max_results: Annotated[int, "Maximum number of results to return"] = 5,
) -> str:
    """Search the web for current information using Tavily and return a formatted summary of results."""
    response = tavily_client.search(
        query=query,
        search_depth="basic",
        max_results=max_results,
        include_answer=True,
    )

    lines = []
    if response.get("answer"):
        lines.append(f"Answer: {response['answer']}")
    for r in response.get("results", []):
        lines.append(f"- {r['title']} ({r['url']}): {r['content']}")

    return "\n".join(lines)

web_search

FunctionTool(name='web_search', description='Search the web for current information using Tavily and return a formatted summary of results.', params_json_schema={'properties': {'query': {'description': 'The search query to look up on the web', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'description': 'Maximum number of results to return', 'title': 'Max Results', 'type': 'integer'}}, 'required': ['query', 'max_results'], 'title': 'web_search_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000260A42B0CD0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

## Smoke test

`@function_tool` wraps `web_search` into a `FunctionTool`, whose `on_invoke_tool` is async and expects a `RunContextWrapper` plus JSON-string args — not convenient for a quick manual check, and would require running it through an `Agent`.

Instead, this calls the underlying Tavily client directly with a sample query to confirm the API key and connectivity work end-to-end, **without creating any agent**.

In [3]:
sample = tavily_client.search(
    query="latest news on OpenAI Agents SDK",
    max_results=3,
    include_answer=True,
)

print(sample.get("answer"))
for r in sample.get("results", []):
    print("-", r["title"], r["url"])

The latest update to the OpenAI Agents SDK enhances sandbox capabilities and introduces new tools for safer, controlled agent operations. It now supports TypeScript and includes features for long-horizon tasks. The SDK continues to evolve with more agent capabilities planned.
- Releases · openai/openai-agents-python https://github.com/openai/openai-agents-python/releases
- OpenAI updates its Agents SDK to help enterprises build safer, more ... https://techcrunch.com/2026/04/15/openai-updates-its-agents-sdk-to-help-enterprises-build-safer-more-capable-agents
- The next evolution of the Agents SDK https://openai.com/index/the-next-evolution-of-the-agents-sdk


# First Agent: Researcher

The **Researcher** agent gathers facts on a topic using the `web_search` tool via tool calling. It is strictly scoped to research: no analysis, trend detection, or insight generation. Its structured output (`ResearchOutput`) is the contract a future **Analyst** agent will consume.

In [4]:
from pydantic import BaseModel, Field


class ResearchRequest(BaseModel):
    topic: str = Field(description="The user query or topic to research")
    max_results: int = Field(default=5, ge=1, le=10, description="Max search results to gather")


class ResearchSource(BaseModel):
    title: str = Field(description="Title of the source")
    url: str = Field(description="URL of the source")
    snippet: str = Field(description="Relevant excerpt/content from the source")


class ResearchOutput(BaseModel):
    topic: str = Field(description="The topic that was researched")
    key_facts: list[str] = Field(description="Concise factual findings, one per item — no analysis or interpretation")
    sources: list[ResearchSource] = Field(description="Sources used to gather the findings")


In [5]:
from agents import Agent, Runner

RESEARCHER_INSTRUCTIONS = """
You are a Researcher agent. Your sole job is to gather factual information on a
given topic using the web_search tool.

Rules:
- Always use the web_search tool to gather up-to-date information before answering.
- Report only facts you found — do not analyze, interpret, identify trends, assess
  risk, or draw conclusions. That is a separate agent's job.
- key_facts: concise, individually-verifiable factual statements, one per item.
- sources: title, url, and a short relevant snippet for every source used.
""".strip()

researcher_agent = Agent(
    name="Researcher",
    instructions=RESEARCHER_INSTRUCTIONS,
    model="gpt-5-mini",
    tools=[web_search],
    output_type=ResearchOutput,
)


In [6]:
request = ResearchRequest(topic="Recent developments in the OpenAI Agents SDK")

result = await Runner.run(researcher_agent, request.topic)
research_output: ResearchOutput = result.final_output

print(research_output.model_dump_json(indent=2))


{
  "topic": "Recent developments in the OpenAI Agents SDK",
  "key_facts": [
    "OpenAI Agents SDK release notes list version 0.22.0 in the breaking change changelog.",
    "Release 0.21.1 was published with documentation updates (docs: updates for v0.21.0) on the Agents SDK GitHub releases.",
    "Version 0.20.0 included a potentially breaking MCP dependency migration and updated the SDK default model used when an agent or run does not explicitly select one.",
    "Version 0.10.0 added websocket transport support for the Responses API.",
    "OpenAI announced 'New tools for building agents' on March 11, 2025, describing new tooling and integrations for agent workflows.",
    "Documentation was updated to include hosted multi-agent support and runnable UserContext examples.",
    "A community and third‑party coverage describe the Agents SDK as launched in March 2025 as a production-ready evolution of earlier experimental projects.",
    "The o1 model release changed messaging roles: 

# Second Agent: Analyst

The **Analyst** agent has no tools. It consumes the Researcher's `ResearchOutput` (facts + sources) and extracts key trends, risks, and insights, writing exactly two paragraphs. Its own output type, `AnalystOutput`, follows the same design pattern as `ResearchOutput` (a `topic` field plus one content field) so the pipeline stays structurally consistent.

In [7]:
class AnalystOutput(BaseModel):
    topic: str = Field(description="The topic that was analyzed")
    analysis: str = Field(description="Exactly two paragraphs covering key trends, risks, and insights drawn from the research notes")


In [8]:
ANALYST_INSTRUCTIONS = """
You are an Analyst agent. You receive factual research notes and extract key
trends, risks, and insights from them. You have no tools — work only from
the research notes provided in the input, do not invent facts.

Output exactly two paragraphs in the `analysis` field:
- Paragraph 1: key trends observed in the research notes.
- Paragraph 2: risks and other notable insights.
""".strip()

analyst_agent = Agent(
    name="Analyst",
    instructions=ANALYST_INSTRUCTIONS,
    model="gpt-5-mini",
    tools=[],
    output_type=AnalystOutput,
)


In [9]:
analyst_input = research_output.model_dump_json()

analyst_result = await Runner.run(analyst_agent, analyst_input)
analyst_output: AnalystOutput = analyst_result.final_output

print(analyst_output.model_dump_json(indent=2))


{
  "topic": "Recent developments in the OpenAI Agents SDK",
  "analysis": "The notes show rapid iteration and maturation of the Agents SDK since its March 2025 launch: multiple releases (0.10.0, 0.20.0, 0.21.1, and a 0.22.0 entry in the breaking-change changelog) added transport and platform features (websocket transport for Responses API in 0.10.0), hosted multi-agent support, runnable UserContext examples, and documentation updates; OpenAI’s March 11, 2025 announcement and community writeups frame the SDK as a production-ready evolution of earlier experimental work. There are also behavioral-default changes: 0.20.0 updated the SDK default model when none is specified, and the o1 model release shifted messaging defaults from a deprecated \"system\" role to a \"developer\" role in examples and docs.\n\nKey risks and notable insights center on migration and compatibility: the 0.20.0 potentially breaking MCP dependency migration and the presence of breaking changes in the 0.22.0 changel

# Third Agent: Writer, and the Full Pipeline

The **Writer** agent has no tools. It takes the Analyst's `AnalystOutput` and turns it into a polished Markdown research report — the final, human-readable deliverable, so it has no `output_type` (free-form text, nothing downstream needs to parse it further).

`manager_run(user_query)` chains **Researcher → Analyst → Writer**, sharing memory across all three agents for that run via a single `SQLiteSession`.

In [10]:
WRITER_INSTRUCTIONS = """
You are a Writer agent. You receive an analyst's trends/risks/insights analysis
and turn it into a polished, well-organized research report. You have no tools
— work only from the analysis provided, do not invent facts.

Write the report in Markdown with a short title, a brief introduction, and
clearly organized sections (e.g. Overview, Key Trends, Risks & Insights,
Conclusion).
""".strip()

writer_agent = Agent(
    name="Writer",
    instructions=WRITER_INSTRUCTIONS,
    model="gpt-5-mini",
    tools=[],
    # No output_type: this is the final human-readable deliverable, free-form Markdown
)


In [11]:
import uuid
from agents import SQLiteSession


async def manager_run(user_query: str) -> str:
    """Runs Researcher -> Analyst -> Writer for one query, sharing memory
    across all three agents via a single SQLiteSession for this run."""
    session = SQLiteSession(session_id=f"manager_run_{uuid.uuid4().hex}")

    research_result = await Runner.run(researcher_agent, user_query, session=session)
    research_output: ResearchOutput = research_result.final_output

    analyst_result = await Runner.run(
        analyst_agent, research_output.model_dump_json(), session=session
    )
    analyst_output: AnalystOutput = analyst_result.final_output

    writer_result = await Runner.run(
        writer_agent, analyst_output.model_dump_json(), session=session
    )
    final_report: str = writer_result.final_output

    return final_report


In [12]:
from IPython.display import Markdown, display

final_report = await manager_run("Recent developments in the OpenAI Agents SDK")
display(Markdown(final_report))


# Recent developments in the OpenAI Agents SDK

Introduction  
This brief report summarizes recent product and ecosystem developments for the OpenAI Agents SDK and highlights core trends, risks, and operational insights for teams evaluating or adopting the SDK.

Overview
- Launched March 2025 as a production-ready evolution of OpenAI’s experimental “Swarm” work.
- Designed as a minimalist agent framework exposing a small set of core primitives and a harness for real-world agent workflows.
- Initial feature rollout prioritized Python; enhanced harness features (configurable memory, sandbox-aware orchestration, filesystem tools, tracing, guardrails, standardized integrations) shipped first for Python.
- An official TypeScript/JavaScript package (@openai/agents) was later published (community-reported around July 3, 2025) supporting Node.js, Deno, and Bun; it includes sandbox APIs and Realtime/RealtimeAgent support.
- Active development and releases continue on GitHub, with non-breaking minor updates and provider-neutral testing APIs.
- Ecosystem integrations are emerging (example: Temporal integration announced July 2025 with a GA update reported March 2026), plus community tutorials and discussion driving adoption.

Key trends
- Python-first, multi-language expansion: feature-rich Python harness followed by an official TypeScript SDK to meet broader developer needs.
- Minimalist core + expanding harness: small primitive set plus added capabilities for memory, filesystems, sandboxes, and orchestration to support production workflows.
- Safety-by-design emphasis: native sandbox execution and guardrails intended to limit unsafe behavior when agents interact with files, systems, or the web.
- Realtime/browser agent support: native realtime APIs in the SDK (browser and server) to power interactive agent experiences.
- Active ecosystem growth: GitHub activity, community tooling, and third-party integrations (e.g., Temporal) accelerating adoption but increasing surface area.

Risks & insights
- Security & operational surface area: sandboxing and guardrails reduce risk but add operational complexity; safe execution remains a central, unresolved operational challenge.
- Fragmentation & adoption friction: Python-first rollout then later TypeScript release produced short-term platform fragmentation and some community timeline confusion for multi-platform teams.
- Compatibility & maintenance risk: rapid expansion of features and integrations increases the chance of API churn or subtle incompatibilities despite OpenAI’s emphasis on non-breaking minor releases and provider-neutral testing.
- Runtime complexity for realtime and orchestrated scenarios: browser realtime agents and orchestration with systems like Temporal introduce latency, state-management, and observability demands that teams must plan for.
- Trust & validation: reliance on SDK guardrails and sandboxing requires careful validation and monitoring; organizations should not assume sandbox guarantees eliminate all risks.

Conclusion
The Agents SDK is maturing quickly from research prototype to a production-oriented platform: core primitives plus an expanding harness make it more capable for file- and system-oriented agent workflows, and language support now covers Python and TypeScript/JavaScript. Teams should evaluate the SDK’s sandbox and guardrail features closely, plan for operational complexity (realtime, orchestration, and multi-language support), and track release notes and ecosystem integrations to manage compatibility and security risks.